# 📊 Unemployment Analysis in India (2019–2020)
### Impact of COVID-19 on India's Labour Market

> **Internship Project — CodeAlpha | Data Science Track**  
> **Task 2:** Unemployment Analysis with Python

---

### 🎯 Objectives
1. Analyze unemployment rate data across Indian states (2019–2020)
2. Identify the impact of **COVID-19 lockdown** on unemployment
3. Explore **Rural vs Urban** unemployment disparities
4. Uncover **seasonal & regional trends** in labour participation
5. Derive insights for **economic and social policy** recommendations

---

### 📁 Datasets Used
| Dataset | Period | Rows | Features |
|---|---|---|---|
| `Unemployment_in_India.csv` | May 2019 – Feb 2020 | 768 | Region, Date, Frequency, Unemployment %, Employed, LPR, Area |
| `Unemployment_Rate_upto_11_2020.csv` | Jan 2020 – Oct 2020 | 267 | Same + Geo-coordinates (lat/lon) |


In [ ]:
# ─── Imports & Configuration ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dark professional theme
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

ACCENT   = '#58a6ff'
ACCENT2  = '#f85149'
ACCENT3  = '#3fb950'
ACCENT4  = '#d2a8ff'
PALETTE  = [ACCENT, ACCENT2, ACCENT3, ACCENT4, '#ffa657', '#79c0ff', '#56d364']

print("✅ Libraries loaded successfully!")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   seaborn : {sns.__version__}")


## 1️⃣ Data Loading & Initial Exploration

In [ ]:
# ─── Load Datasets ──────────────────────────────────────────────────────────
df1 = pd.read_csv('data/Unemployment_in_India.csv')
df2 = pd.read_csv('data/Unemployment_Rate_upto_11_2020.csv')

# Strip whitespace from column names
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# Parse dates
df1['Date'] = pd.to_datetime(df1['Date'].str.strip(), dayfirst=True, errors='coerce')
df2['Date'] = pd.to_datetime(df2['Date'].str.strip(), dayfirst=True, errors='coerce')

print("=" * 55)
print("📂 Dataset 1: Unemployment_in_India.csv")
print("=" * 55)
print(f"  Shape    : {df1.shape[0]} rows × {df1.shape[1]} columns")
print(f"  States   : {df1['Region'].nunique()} unique regions")
print(f"  Period   : {df1['Date'].min().date()} → {df1['Date'].max().date()}")
print(f"  Areas    : {df1['Area'].unique().tolist()}")
print()
print("=" * 55)
print("📂 Dataset 2: Unemployment_Rate_upto_11_2020.csv")
print("=" * 55)
print(f"  Shape    : {df2.shape[0]} rows × {df2.shape[1]} columns")
print(f"  States   : {df2['Region'].nunique()} unique regions")
print(f"  Period   : {df2['Date'].min().date()} → {df2['Date'].max().date()}")


In [ ]:
# ─── Preview Dataset 1 ──────────────────────────────────────────────────────
print("\n📊 Dataset 1 — First 5 rows:")
display(df1.head())
print("\n📋 Data Types & Null Values:")
display(pd.DataFrame({
    'dtype': df1.dtypes,
    'non_null': df1.count(),
    'null_%': (df1.isnull().mean() * 100).round(2)
}))


In [ ]:
# ─── Preview Dataset 2 ──────────────────────────────────────────────────────
print("\n📊 Dataset 2 — First 5 rows:")
display(df2.head())
print("\n📋 Statistical Summary — Dataset 2:")
display(df2.describe().round(2))


## 2️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# ─── Key Statistics ────────────────────────────────────────────────────────
unemp_col1 = 'Estimated Unemployment Rate (%)'
unemp_col2 = 'Estimated Unemployment Rate (%)'
empl_col1  = 'Estimated Employed'
empl_col2  = 'Estimated Employed'
lpr_col1   = 'Estimated Labour Participation Rate (%)'
lpr_col2   = 'Estimated Labour Participation Rate (%)'

pre_covid  = df2[df2['Date'] < '2020-03-24']
post_covid = df2[df2['Date'] >= '2020-03-24']

print("📈 KEY STATISTICS SUMMARY")
print("=" * 55)
print(f"  Overall avg unemployment (2019-2020) : {df1[unemp_col1].mean():.2f}%")
print(f"  Pre-COVID avg  (Jan–Mar 2020)         : {pre_covid[unemp_col2].mean():.2f}%")
print(f"  Post-COVID avg (Apr–Oct 2020)         : {post_covid[unemp_col2].mean():.2f}%")
print(f"  Peak unemployment month               : {df2.groupby('Date')[unemp_col2].mean().idxmax().strftime('%B %Y')}")
print(f"  Peak unemployment rate                : {df2.groupby('Date')[unemp_col2].mean().max():.2f}%")
print(f"  Highest state unemployment            : {df2.groupby('Region')[unemp_col2].mean().idxmax()}")
print(f"  Lowest state unemployment             : {df2.groupby('Region')[unemp_col2].mean().idxmin()}")
if 'Area' in df1.columns:
    rural_avg = df1[df1['Area']=='Rural'][unemp_col1].mean()
    urban_avg = df1[df1['Area']=='Urban'][unemp_col1].mean()
    print(f"  Rural avg unemployment                : {rural_avg:.2f}%")
    print(f"  Urban avg unemployment                : {urban_avg:.2f}%")


## 3️⃣ National Unemployment Trend (2019–2020)

In [ ]:
# ─── National Trend with COVID Annotation ──────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

national1 = df1.groupby('Date')[unemp_col1].mean().reset_index()
national2 = df2.groupby('Date')[unemp_col2].mean().reset_index()
combined  = pd.concat([national1, national2]).drop_duplicates('Date').sort_values('Date')

ax.fill_between(combined['Date'], combined[unemp_col1], alpha=0.15, color=ACCENT)
ax.plot(combined['Date'], combined[unemp_col1], color=ACCENT, lw=2.5, label='Avg Unemployment Rate (%)')

covid_val = combined.loc[combined['Date'] >= '2020-04-01', unemp_col1].max()
ax.axvline(pd.Timestamp('2020-03-24'), color=ACCENT2, ls='--', lw=1.5, alpha=0.8)
ax.annotate('COVID-19 Lockdown\n(Mar 24, 2020)',
            xy=(pd.Timestamp('2020-03-24'), covid_val * 0.95),
            xytext=(pd.Timestamp('2019-11-01'), covid_val * 1.05),
            color=ACCENT2, fontsize=9.5,
            arrowprops=dict(arrowstyle='->', color=ACCENT2, lw=1.5))

ax.set_title('India Unemployment Rate Trend (2019–2020)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Date'); ax.set_ylabel('Unemployment Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.grid(True, ls='--', alpha=0.4); ax.legend()
plt.tight_layout(); plt.show()


## 4️⃣ State-wise Unemployment Analysis

In [ ]:
# ─── State-wise Average Unemployment ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
state_avg = df1.groupby('Region')[unemp_col1].mean().sort_values(ascending=True)
colors = [ACCENT2 if v > state_avg.mean() else ACCENT for v in state_avg]
bars = ax.barh(state_avg.index, state_avg.values, color=colors, edgecolor='none', height=0.7)
ax.axvline(state_avg.mean(), color='#ffa657', ls='--', lw=1.5, label=f'National Avg ({state_avg.mean():.1f}%)')
for bar, val in zip(bars, state_avg.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8.5, color='#e6edf3')

ax.set_title('Average Unemployment Rate by State (2019–2020)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Average Unemployment Rate (%)'); ax.legend()
ax.grid(axis='x', ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

print("\n🔴 States ABOVE national average (high unemployment):")
above = state_avg[state_avg > state_avg.mean()].sort_values(ascending=False)
for s, v in above.items(): print(f"   {s}: {v:.2f}%")


## 5️⃣ Rural vs Urban Unemployment

In [ ]:
# ─── Rural vs Urban Analysis ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

if 'Area' in df1.columns:
    for area, color in zip(['Rural', 'Urban'], [ACCENT3, ACCENT2]):
        sub = df1[df1['Area'] == area].groupby('Date')[unemp_col1].mean()
        axes[0].plot(sub.index, sub.values, color=color, lw=2.2, label=area)
        axes[0].fill_between(sub.index, sub.values, alpha=0.1, color=color)
    axes[0].axvline(pd.Timestamp('2020-03-24'), color='#ffa657', ls='--', lw=1.2, alpha=0.8)
    axes[0].set_title('Rural vs Urban Unemployment Over Time', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Date'); axes[0].set_ylabel('Unemployment Rate (%)')
    axes[0].legend(); axes[0].grid(True, ls='--', alpha=0.4)

    rural_data = df1[df1['Area'] == 'Rural'][unemp_col1].dropna()
    urban_data = df1[df1['Area'] == 'Urban'][unemp_col1].dropna()
    bp = axes[1].boxplot([rural_data, urban_data], labels=['Rural', 'Urban'],
                         patch_artist=True, widths=0.5,
                         medianprops=dict(color='white', lw=2),
                         whiskerprops=dict(color='#8b949e'), capprops=dict(color='#8b949e'),
                         flierprops=dict(marker='o', color='#8b949e', markersize=4))
    for patch, color in zip(bp['boxes'], [ACCENT3, ACCENT2]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    axes[1].set_title('Distribution: Rural vs Urban', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Unemployment Rate (%)')
    axes[1].grid(axis='y', ls='--', alpha=0.4)

plt.suptitle('Rural vs Urban Unemployment Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

if 'Area' in df1.columns:
    print(f"\n📊 Rural Mean: {rural_data.mean():.2f}% | Median: {rural_data.median():.2f}%")
    print(f"   Urban Mean: {urban_data.mean():.2f}% | Median: {urban_data.median():.2f}%")


## 6️⃣ COVID-19 Impact Analysis

In [ ]:
# ─── COVID Impact: Before vs During ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

monthly = df2.groupby('Date')[unemp_col2].mean().reset_index()
monthly['Phase'] = monthly['Date'].apply(lambda d: 'During COVID' if d >= pd.Timestamp('2020-03-24') else 'Pre-COVID')

colors_phase = {'Pre-COVID': ACCENT3, 'During COVID': ACCENT2}
for phase, grp in monthly.groupby('Phase'):
    axes[0].bar(grp['Date'], grp[unemp_col2], color=colors_phase[phase], label=phase, alpha=0.85, width=20)
axes[0].set_title('Monthly Unemployment: Pre vs During COVID', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Unemployment Rate (%)')
axes[0].legend(); axes[0].grid(axis='y', ls='--', alpha=0.4)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

pre  = df2[df2['Date'] < pd.Timestamp('2020-03-24')].groupby('Region')[unemp_col2].mean()
post = df2[df2['Date'] >= pd.Timestamp('2020-03-24')].groupby('Region')[unemp_col2].mean()
impact = (post - pre).dropna().sort_values(ascending=False).head(12)
colors_imp = [ACCENT2 if v > 0 else ACCENT3 for v in impact.values]
axes[1].barh(impact.index, impact.values, color=colors_imp, edgecolor='none')
axes[1].axvline(0, color='white', lw=0.8)
axes[1].set_title('COVID Impact by State\n(Post − Pre Lockdown)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Change in Unemployment Rate (pp)')
axes[1].grid(axis='x', ls='--', alpha=0.4)

plt.tight_layout(); plt.show()

print("\n⚠️  COVID-19 Impact Summary:")
print(f"  Pre-COVID avg  : {pre.mean():.2f}%")
print(f"  Post-COVID avg : {post.mean():.2f}%")
print(f"  Increase       : +{(post.mean() - pre.mean()):.2f} percentage points")
print(f"\n  Most affected states:")
for s, v in impact.head(5).items():
    print(f"    {s}: +{v:.2f} pp")


## 7️⃣ Labour Participation Rate Heatmap

In [ ]:
# ─── LPR Heatmap ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
df1['Month'] = df1['Date'].dt.to_period('M').astype(str)
heat_data = df1.pivot_table(index='Region', columns='Month', values=lpr_col1, aggfunc='mean')
heat_data = heat_data.dropna(thresh=3)

sns.heatmap(heat_data, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#0d1117',
            cbar_kws={'label': 'Labour Participation Rate (%)', 'shrink': 0.8})
ax.set_title('Labour Participation Rate Heatmap by State & Month', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Month'); ax.set_ylabel('State')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout(); plt.show()


## 8️⃣ Top 5 Most COVID-Affected States

In [ ]:
# ─── Top 5 State Timelines ──────────────────────────────────────────────────
pre_   = df2[df2['Date'] < '2020-03-24'].groupby('Region')[unemp_col2].mean()
post_  = df2[df2['Date'] >= '2020-03-24'].groupby('Region')[unemp_col2].mean()
impact_ = (post_ - pre_).dropna().sort_values(ascending=False)
top5   = impact_.head(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
for i, state in enumerate(top5):
    sub = df2[df2['Region'] == state].sort_values('Date')
    ax.plot(sub['Date'], sub[unemp_col2], lw=2.2, label=state, color=PALETTE[i])
    ax.fill_between(sub['Date'], sub[unemp_col2], alpha=0.07, color=PALETTE[i])
ax.axvline(pd.Timestamp('2020-03-24'), color='white', ls='--', lw=1.2, alpha=0.7)
ax.text(pd.Timestamp('2020-03-28'), ax.get_ylim()[1]*0.88, 'Lockdown', color='white', fontsize=9)
ax.set_title('Top 5 COVID-Affected States — Unemployment Timeline', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Date'); ax.set_ylabel('Unemployment Rate (%)')
ax.legend(loc='upper left'); ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout(); plt.show()


## 9️⃣ Employed Workforce Trend

In [ ]:
# ─── Employment Trend ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
emp_trend = df2.groupby('Date')[empl_col2].sum() / 1e7  # in Crores

ax.fill_between(emp_trend.index, emp_trend.values, alpha=0.15, color=ACCENT4)
ax.plot(emp_trend.index, emp_trend.values, color=ACCENT4, lw=2.5)
ax.axvline(pd.Timestamp('2020-03-24'), color=ACCENT2, ls='--', lw=1.5, alpha=0.8)
ax.annotate('Lockdown — Sharp\nJob Loss',
            xy=(pd.Timestamp('2020-04-15'), emp_trend['2020-04':].min()),
            xytext=(pd.Timestamp('2020-06-01'), emp_trend.mean() * 0.85),
            color=ACCENT2, fontsize=9.5,
            arrowprops=dict(arrowstyle='->', color=ACCENT2, lw=1.5))
ax.set_title('Estimated Employed Workforce Over Time (Crores)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Date'); ax.set_ylabel('Employed (Crores)')
ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

print(f"\n📉 Jobs lost at peak (Apr 2020 vs Jan 2020):")
jan = df2[df2['Date']=='2020-01-31'][empl_col2].sum()
apr = df2[df2['Date']=='2020-04-30'][empl_col2].sum()
print(f"   January 2020 : {jan/1e7:.2f} Crore workers")
print(f"   April 2020   : {apr/1e7:.2f} Crore workers")
print(f"   Job loss     : {(jan-apr)/1e7:.2f} Crore ({((jan-apr)/jan*100):.1f}% decline)")


## 🔟 Correlation Analysis

In [ ]:
# ─── Correlation Matrix ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
num_cols = [unemp_col2, empl_col2, lpr_col2]
corr = df2[num_cols].corr()
corr.columns = ['Unemployment\nRate', 'Employed\nWorkforce', 'Labour\nParticipation']
corr.index   = ['Unemployment\nRate', 'Employed\nWorkforce', 'Labour\nParticipation']
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=1, linecolor='#0d1117', square=True,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 13})
ax.set_title('Correlation Matrix — Key Indicators', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout(); plt.show()

print("\n📊 Key Correlations:")
print(f"  Unemployment ↔ Labour Participation : {corr.iloc[0,2]:.2f}")
print(f"  Unemployment ↔ Employed             : {corr.iloc[0,1]:.2f}")
print(f"  Employed     ↔ Labour Participation : {corr.iloc[1,2]:.2f}")


## 📌 Key Findings & Policy Insights

---

### 🔍 Major Findings

| # | Finding | Insight |
|---|---|---|
| 1 | **COVID spike** | National unemployment surged from ~8% to **23.5%** in April 2020 — a 3× jump in 4 weeks |
| 2 | **Urban > Rural** | Urban unemployment was consistently higher; urban informal workers hit hardest |
| 3 | **State disparities** | Haryana, Tripura & Jharkhand had chronically high rates even pre-COVID |
| 4 | **Labour exodus** | ~12+ Crore workers left the active labour force at peak lockdown (April 2020) |
| 5 | **Recovery** | Post-June 2020, rates began declining — but remained well above pre-COVID baseline |
| 6 | **LPR correlation** | Strong positive correlation between Labour Participation Rate and Employed workforce |

---

### 💡 Policy Recommendations

1. **Targeted urban safety nets** — Urban informal sector workers need portable social security
2. **MGNREGA expansion** — Rural work guarantee schemes should be extended to cover urban areas during crises
3. **State-specific interventions** — Haryana, Tripura, Jharkhand need structural job creation programs
4. **Reskilling programs** — Post-COVID labour market shift demands upskilling in digital & services sectors
5. **Real-time data collection** — Monthly granular employment tracking should become national policy

---

### 🛠️ Technical Stack

```
Python 3.x  |  Pandas  |  NumPy  |  Matplotlib  |  Seaborn
```

---

*Project by: [Your Name] | CodeAlpha Data Science Internship | 2024*
